In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder.appName("Spark df operations").enableHiveSupport().getOrCreate()
print("Driver Python:", sys.executable)
print("Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/15 10:07:43 WARN Utils: Your hostname, Darviks-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.5 instead (on interface en0)
26/06/15 10:07:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 10:07:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Driver Python: /Users/darvikkunalbanda/DataEngineering/.venv/bin/python
Spark: 4.1.2


In [2]:
customer_df = spark.read.parquet('/Users/darvikkunalbanda/DataEngineering/DE_Drill/dataset/customers_parquet')
orders_df = spark.read.csv('/Users/darvikkunalbanda/DataEngineering/DE_Drill/dataset/orders_csv',header=True)

In [3]:
df = customer_df.unionAll(customer_df)

In [4]:
df.show(5,False)

+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
|customer_id|name |city    |state|signup_date|tier  |score|last_order_date|active|region|
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
|C001       |Kate |Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |
|C002       |Jack |Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |
|C003       |Leo  |Atlanta |OR   |2024-04-15 |Gold  |928  |2025-01-25     |true  |West  |
|C004       |Frank|Austin  |IL   |2024-04-25 |Gold  |204  |2025-01-13     |true  |East  |
|C005       |Noah |Seattle |TX   |2024-05-26 |Bronze|847  |2025-04-18     |true  |West  |
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
only showing top 5 rows


## using sql

In [5]:
#temp view
df.createOrReplaceTempView('df')

In [6]:
#distinct

spark.sql("SELECT DISTINCT customer_id , name FROM df").show()

+-----------+-------+
|customer_id|   name|
+-----------+-------+
|       C021|    Bob|
|       C024|  Grace|
|       C022|    Leo|
|       C036|    Eve|
|       C003|    Leo|
|       C007|   Kate|
|       C027|  Alice|
|       C015|  Alice|
|       C010|   Noah|
|       C020|   Jack|
|       C048|  Grace|
|       C029|    Ivy|
|       C002|   Jack|
|       C026| Olivia|
|       C038|   Noah|
|       C006|    Bob|
|       C049|   Hank|
|       C040| Olivia|
|       C041|    Eve|
|       C013|Charlie|
+-----------+-------+
only showing top 20 rows


## dataframe

In [7]:
#method 1 - distinct()
dist_df = df.distinct()
dist_df.show()

+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|customer_id|   name|    city|state|signup_date|    tier|score|last_order_date|active|region|
+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|       C044|  Alice|  Boston|   TX| 2024-07-12|    Gold|  546|     2025-05-17|  true|  West|
|       C014|   Kate|  Boston|   GA| 2024-05-25|    Gold|  214|     2025-03-14|  true|  West|
|       C049|   Hank|   Miami|   CO| 2024-09-16|  Silver|  774|     2025-01-10| false| North|
|       C043|  Frank| Atlanta|   OR| 2024-11-27|    Gold|  128|     2025-01-09|  true|  East|
|       C037|Charlie|   Miami|   GA| 2024-12-14|  Bronze|  214|     2025-01-23|  true| North|
|       C050|   Noah|  Denver|   TX| 2024-04-26|  Silver|  250|     2025-01-02|  true|  West|
|       C007|   Kate|  Denver|   TX| 2024-02-28|  Silver|  987|     2025-01-13| false|  West|
|       C039| Olivia| Seattle|   CO| 2024-11-08|  Bronze|  4

In [8]:
#method 2 - drop duplicates()
dedup_df = df.drop_duplicates()
dedup_df.show()

+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|customer_id|   name|    city|state|signup_date|    tier|score|last_order_date|active|region|
+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|       C044|  Alice|  Boston|   TX| 2024-07-12|    Gold|  546|     2025-05-17|  true|  West|
|       C014|   Kate|  Boston|   GA| 2024-05-25|    Gold|  214|     2025-03-14|  true|  West|
|       C049|   Hank|   Miami|   CO| 2024-09-16|  Silver|  774|     2025-01-10| false| North|
|       C043|  Frank| Atlanta|   OR| 2024-11-27|    Gold|  128|     2025-01-09|  true|  East|
|       C037|Charlie|   Miami|   GA| 2024-12-14|  Bronze|  214|     2025-01-23|  true| North|
|       C050|   Noah|  Denver|   TX| 2024-04-26|  Silver|  250|     2025-01-02|  true|  West|
|       C007|   Kate|  Denver|   TX| 2024-02-28|  Silver|  987|     2025-01-13| false|  West|
|       C039| Olivia| Seattle|   CO| 2024-11-08|  Bronze|  4

## handle nulls

    Integer type : fill(0)
    String type : fill("")

In [9]:
#handle nulls
df_nulls = spark.read.csv("/Users/darvikkunalbanda/DataEngineering/DE_Drill/dataset/customers_nulls.csv",header=True,inferSchema=True)
df_nulls.show(5,False)

+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
|customer_id|name |city    |state|signup_date|tier  |score|last_order_date|active|region|
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
|C001       |Kate |Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |
|C002       |Jack |Portland|NY   |2024-01-03 |NULL  |338  |2025-05-20     |true  |South |
|C003       |NULL |Atlanta |OR   |2024-04-15 |Gold  |NULL |2025-01-25     |true  |West  |
|C004       |Frank|Austin  |IL   |NULL       |Gold  |204  |NULL           |true  |NULL  |
|C005       |Noah |Seattle |TX   |2024-05-26 |Bronze|847  |2025-04-18     |NULL  |West  |
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
only showing top 5 rows


In [10]:
df_notnull = df_nulls.na.fill("")
df_notnull.show()

+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|customer_id|   name|    city|state|signup_date|    tier|score|last_order_date|active|region|
+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|       C001|   Kate|  Boston|   NY| 2024-12-09|  Silver|  328|     2025-02-24|  true| North|
|       C002|   Jack|Portland|   NY| 2024-01-03|        |  338|     2025-05-20|  true| South|
|       C003|       | Atlanta|   OR| 2024-04-15|    Gold| NULL|     2025-01-25|  true|  West|
|       C004|  Frank|  Austin|   IL|       NULL|    Gold|  204|           NULL|  true|      |
|       C005|   Noah| Seattle|   TX| 2024-05-26|  Bronze|  847|     2025-04-18|  NULL|  West|
|       C006|    Bob| Atlanta|   TX| 2024-11-20|    Gold|  691|     2025-02-23|  true| North|
|       C007|   Kate|  Denver|   TX|       NULL|  Silver|  987|           NULL| false|  West|
|       C008|   Kate| Seattle|   IL| 2024-06-12|        |  7

In [11]:
#only to fill in required columns

df_notnull_1 = df_nulls.na.fill("2026-01-01",['signup_date'])
df_notnull_1.show()

+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|customer_id|   name|    city|state|signup_date|    tier|score|last_order_date|active|region|
+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|       C001|   Kate|  Boston|   NY| 2024-12-09|  Silver|  328|     2025-02-24|  true| North|
|       C002|   Jack|Portland|   NY| 2024-01-03|    NULL|  338|     2025-05-20|  true| South|
|       C003|   NULL| Atlanta|   OR| 2024-04-15|    Gold| NULL|     2025-01-25|  true|  West|
|       C004|  Frank|  Austin|   IL|       NULL|    Gold|  204|           NULL|  true|  NULL|
|       C005|   Noah| Seattle|   TX| 2024-05-26|  Bronze|  847|     2025-04-18|  NULL|  West|
|       C006|    Bob| Atlanta|   TX| 2024-11-20|    Gold|  691|     2025-02-23|  true| North|
|       C007|   Kate|  Denver|   TX|       NULL|  Silver|  987|           NULL| false|  West|
|       C008|   Kate| Seattle|   IL| 2024-06-12|    NULL|  7

In [12]:
df_nulls.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- tier: string (nullable = true)
 |-- score: integer (nullable = true)
 |-- last_order_date: date (nullable = true)
 |-- active: boolean (nullable = true)
 |-- region: string (nullable = true)



In [13]:
df_notnull_2 = df_nulls.na.fill(0)
df_notnull_2.show()

+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|customer_id|   name|    city|state|signup_date|    tier|score|last_order_date|active|region|
+-----------+-------+--------+-----+-----------+--------+-----+---------------+------+------+
|       C001|   Kate|  Boston|   NY| 2024-12-09|  Silver|  328|     2025-02-24|  true| North|
|       C002|   Jack|Portland|   NY| 2024-01-03|    NULL|  338|     2025-05-20|  true| South|
|       C003|   NULL| Atlanta|   OR| 2024-04-15|    Gold|    0|     2025-01-25|  true|  West|
|       C004|  Frank|  Austin|   IL|       NULL|    Gold|  204|           NULL|  true|  NULL|
|       C005|   Noah| Seattle|   TX| 2024-05-26|  Bronze|  847|     2025-04-18|  NULL|  West|
|       C006|    Bob| Atlanta|   TX| 2024-11-20|    Gold|  691|     2025-02-23|  true| North|
|       C007|   Kate|  Denver|   TX|       NULL|  Silver|  987|           NULL| false|  West|
|       C008|   Kate| Seattle|   IL| 2024-06-12|    NULL|  7

how to create new columns in df

how to drop existing columns in df

how to rename existing columns in df

In [14]:
customer_df.show(5,False)

+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
|customer_id|name |city    |state|signup_date|tier  |score|last_order_date|active|region|
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
|C001       |Kate |Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |
|C002       |Jack |Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |
|C003       |Leo  |Atlanta |OR   |2024-04-15 |Gold  |928  |2025-01-25     |true  |West  |
|C004       |Frank|Austin  |IL   |2024-04-25 |Gold  |204  |2025-01-13     |true  |East  |
|C005       |Noah |Seattle |TX   |2024-05-26 |Bronze|847  |2025-04-18     |true  |West  |
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
only showing top 5 rows


In [15]:
#creating new col
#method 1 LIT
from pyspark.sql.functions import lit,col
new_df = df.withColumn("country",lit("INDIA"))
new_df.show(3,False)

+-----------+----+--------+-----+-----------+------+-----+---------------+------+------+-------+
|customer_id|name|city    |state|signup_date|tier  |score|last_order_date|active|region|country|
+-----------+----+--------+-----+-----------+------+-----+---------------+------+------+-------+
|C001       |Kate|Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |INDIA  |
|C002       |Jack|Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |INDIA  |
|C003       |Leo |Atlanta |OR   |2024-04-15 |Gold  |928  |2025-01-25     |true  |West  |INDIA  |
+-----------+----+--------+-----+-----------+------+-----+---------------+------+------+-------+
only showing top 3 rows


In [16]:
#method 2 from old_col transforming to new col
new_df_1 = df.withColumn("new_score_pct",col("score")/100)
new_df_1.show(3,False)

+-----------+----+--------+-----+-----------+------+-----+---------------+------+------+-------------+
|customer_id|name|city    |state|signup_date|tier  |score|last_order_date|active|region|new_score_pct|
+-----------+----+--------+-----+-----------+------+-----+---------------+------+------+-------------+
|C001       |Kate|Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |3.28         |
|C002       |Jack|Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |3.38         |
|C003       |Leo |Atlanta |OR   |2024-04-15 |Gold  |928  |2025-01-25     |true  |West  |9.28         |
+-----------+----+--------+-----+-----------+------+-----+---------------+------+------+-------------+
only showing top 3 rows


### how to drop existing columns in dataframe

### how to rename existing columns in dataframe

In [17]:
customer_df.show(5,False)

+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
|customer_id|name |city    |state|signup_date|tier  |score|last_order_date|active|region|
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
|C001       |Kate |Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |
|C002       |Jack |Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |
|C003       |Leo  |Atlanta |OR   |2024-04-15 |Gold  |928  |2025-01-25     |true  |West  |
|C004       |Frank|Austin  |IL   |2024-04-25 |Gold  |204  |2025-01-13     |true  |East  |
|C005       |Noah |Seattle |TX   |2024-05-26 |Bronze|847  |2025-04-18     |true  |West  |
+-----------+-----+--------+-----+-----------+------+-----+---------------+------+------+
only showing top 5 rows


In [18]:
#drop column

drop_df = customer_df.drop("region").drop("state")
drop_df.show(5,False)

+-----------+-----+--------+-----------+------+-----+---------------+------+
|customer_id|name |city    |signup_date|tier  |score|last_order_date|active|
+-----------+-----+--------+-----------+------+-----+---------------+------+
|C001       |Kate |Boston  |2024-12-09 |Silver|328  |2025-02-24     |true  |
|C002       |Jack |Portland|2024-01-03 |Silver|338  |2025-05-20     |true  |
|C003       |Leo  |Atlanta |2024-04-15 |Gold  |928  |2025-01-25     |true  |
|C004       |Frank|Austin  |2024-04-25 |Gold  |204  |2025-01-13     |true  |
|C005       |Noah |Seattle |2024-05-26 |Bronze|847  |2025-04-18     |true  |
+-----------+-----+--------+-----------+------+-----+---------------+------+
only showing top 5 rows


In [19]:
#rename column

rm_col = df.withColumnRenamed("name","customer_name")
rm_col.show(5,False)

+-----------+-------------+--------+-----+-----------+------+-----+---------------+------+------+
|customer_id|customer_name|city    |state|signup_date|tier  |score|last_order_date|active|region|
+-----------+-------------+--------+-----+-----------+------+-----+---------------+------+------+
|C001       |Kate         |Boston  |NY   |2024-12-09 |Silver|328  |2025-02-24     |true  |North |
|C002       |Jack         |Portland|NY   |2024-01-03 |Silver|338  |2025-05-20     |true  |South |
|C003       |Leo          |Atlanta |OR   |2024-04-15 |Gold  |928  |2025-01-25     |true  |West  |
|C004       |Frank        |Austin  |IL   |2024-04-25 |Gold  |204  |2025-01-13     |true  |East  |
|C005       |Noah         |Seattle |TX   |2024-05-26 |Bronze|847  |2025-04-18     |true  |West  |
+-----------+-------------+--------+-----+-----------+------+-----+---------------+------+------+
only showing top 5 rows


### dataframe to file

customer_df = spark.read.parquet('/Users/darvikkunalbanda/DataEngineering/DE_Drill/dataset/customers_parquet')

orders_df = spark.read.csv('/Users/darvikkunalbanda/DataEngineering/DE_Drill/dataset/orders_csv',header=True)

In [22]:
orders_df.rdd.getNumPartitions()

1

26/06/15 12:10:26 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 140626 ms exceeds timeout 120000 ms
26/06/15 12:10:26 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/15 12:10:35 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$